## Raw AROME Dataset Exploration

*Exploratory analysis of the raw AROME forecast dataset.*

This notebook presents an initial exploration of the raw AROME dataset before any cleaning or preprocessing step. The objective is to understand the structure of the dataset, verify its integrity, inspect the variables, and identify potential data quality issues that may affect subsequent processing and model training.

In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [15]:
df = pd.read_csv("../src/Arome/Datasetfinal.csv") # Arôme dataset with problems (missing values, duplicates, etc.)
clean = pd.read_csv("../src/Preparing/Arome_clean_final.csv") # Arôme dataset with fixed problems (missing values, duplicates, etc.)
df.head(1)

,station,datetime,longitude,latitude,u10,v10,t2m,rh2m,u850,v850,u950,v950,psurf,u_gust60,v_gust60,tke20m,edr20m,pblh
0,60033.0,2021-01-01 00:00:00,-13.2,27.15,-0.375698,-2.860277,286.744537,0.758287,-0.529859,-11.777529,-2.984562,-9.461304,101380.125,-0.378971,-2.863943,0.000001,0.000001,10.0


## Exploration of AROME Data Received from the DGM (Ministry of Equipment and Water | Directorate of General Meteorology)

## 1) Duplicate Records Analysis

In [5]:
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Rows    : 1,139,968
Columns : 18


In [ ]:
df.duplicated().sum() # Check for duplicates in the dataset

np.int64(6097)

In [ ]:
duplicates = df[df.duplicated(keep=False)]
duplicates.sort_values(
    by=["station", "datetime"]
).head(10) # Display the first 10 duplicates in the dataset, sorted by station and datetime

,station,datetime,longitude,latitude,u10,v10,t2m,rh2m,u850,v850,u950,v950,psurf,u_gust60,v_gust60,tke20m,edr20m,pblh
25873,60033.0,2021-02-12 00:00:00,-1.95,32.55,1.916237,2.340259,283.798584,0.614084,5.407262,5.275643,1.841847,2.238847,88497.53516,1.920721,2.345119,0.000001,0.000001,10.000000
25874,60033.0,2021-02-12 00:00:00,-1.95,32.55,1.916237,2.340259,283.798584,0.614084,5.407262,5.275643,1.841847,2.238847,88497.53516,1.920721,2.345119,0.000001,0.000001,10.000000
1384,60033.0,2022-01-02 19:00:00,-13.20,27.15,-1.169518,-4.346401,294.513916,0.417736,-8.085458,-2.529974,-6.900595,-3.189829,101587.31250,-0.770452,-7.738855,0.306703,0.005460,131.187499
229468,60033.0,2022-01-02 19:00:00,-13.20,27.15,-1.169518,-4.346401,294.513916,0.417736,-8.085458,-2.529974,-6.900595,-3.189829,101587.31250,-0.770452,-7.738855,0.306703,0.005460,131.187499
9497,60033.0,2022-01-15 19:00:00,-13.20,27.15,-4.579706,2.492281,292.866699,0.153383,-7.503135,12.085605,-10.093262,8.374570,101217.94530,-9.229218,5.517585,0.370362,0.007104,1195.562499
237580,60033.0,2022-01-15 19:00:00,-13.20,27.15,-4.579706,2.492281,292.866699,0.153383,-7.503135,12.085605,-10.093262,8.374570,101217.94530,-9.229218,5.517585,0.370362,0.007104,1195.562499
12421,60033.0,2022-01-20 13:00:00,-13.20,27.15,-5.618690,7.066391,298.768799,0.117959,5.101950,8.238735,-6.708069,11.533584,100942.22660,-8.645630,10.678763,1.420899,0.030320,842.499999
240544,60033.0,2022-01-20 13:00:00,-13.20,27.15,-5.618690,7.066391,298.768799,0.117959,5.101950,8.238735,-6.708069,11.533584,100942.22660,-8.645630,10.678763,1.420899,0.030320,842.499999
17972,60033.0,2022-01-29 11:00:00,-13.20,27.15,0.499817,3.575384,293.026856,0.555708,1.040924,3.682048,0.147995,4.000065,101205.04300,1.279678,6.095566,0.449220,0.005433,455.999999
246108,60033.0,2022-01-29 11:00:00,-13.20,27.15,0.499817,3.575384,293.026856,0.555708,1.040924,3.682048,0.147995,4.000065,101205.04300,1.279678,6.095566,0.449220,0.005433,455.999999


In [8]:
df = df.drop_duplicates(keep="first")
df.duplicated().sum()

np.int64(0)

### Duplicate Removal
The exploration revealed the presence of fully duplicated records. Since all duplicated rows contain identical values across every feature, only the first occurrence of each duplicate was retained. This operation removes redundant observations without affecting the information contained in the dataset.

## 2) Station Coordinate Consistency

In [ ]:
df["station"].unique() # Check the unique stations in the dataset

array([60033., 60060., 60096., 60101., 60107., 60115., 60120., 60135.,
       60136., 60141., 60150., 60155., 60156., 60160., 60191., 60200.,
       60210., 60220., 60230., 60250., 60252., 60265., 60280., 60285.,
       60338., 60340.,    nan])

The following station identifiers were found in the AROME dataset provided by the DGM

The `NaN` values correspond to rows that begin with a date and do not contain a station identifier. These rows were intentionally preserved because their longitude and latitude coordinates will later be used to determine the station to which each observation belongs.


In [ ]:
code_OACI = {
    60033: "GMML",
    60060: "GMMF",
    60096: "GMMH",
    60101: "GMTT",
    60107: "GMTA",
    60115: "GMFO",
    60120: "GMMP",
    60135: "GMME",
    60136: "GMSL",
    60141: "GMFF",
    60150: "GMFM",
    60155: "GMMC",
    60156: "GMMN",
    60160: "GMFI",
    60191: "GMMD",
    60200: "GMFB",
    60210: "GMFK",
    60220: "GMMI",
    60230: "GMMX",
    60250: "GMAA",
    60252: "GMAD",
    60265: "GMMZ",
    60280: "GMAG",
    60285: "GMAT",
    60340: "GMMW"
} # Dictionary mapping station codes to their corresponding OACI codes

In [ ]:
station_coordinates = (
    df.groupby("station")[["longitude", "latitude"]]
      .nunique()
)

station_coordinates # Display the unique longitude and latitude for each station in the dataset

,longitude,latitude
station,,
60033.0,27,27
60060.0,27,27
60096.0,27,27
60101.0,27,27
60107.0,27,27
60115.0,27,27
60120.0,27,27
60135.0,27,27
60136.0,27,27


### Resolving Inconsistent Station Coordinates

The table above shows that each station identifier is associated with multiple longitude and latitude values. More specifically, each station contains 27 distinct longitude values and 27 distinct latitude values.

This is a major data-quality issue because a meteorological station is expected to have a fixed geographical location. Therefore, each station should normally be associated with one consistent longitude–latitude pair.

These coordinate variations may be caused by duplicated records, formatting issues, shifted rows, or observations extracted from nearby AROME grid points.

To assign a representative geographical position to each station, the most frequently occurring coordinate pair will be selected for every station. In other words, the longitude–latitude combination with the highest number of occurrences within each station group will be considered the station’s reference location.

The procedure consists of the following steps:

1. Group the observations by station identifier, longitude, and latitude.
2. Count the number of occurrences of each longitude–latitude pair.
3. For each station, identify the coordinate pair with the highest frequency.
4. Assign this most frequent pair as the representative coordinates of the station.
5. Use these reference coordinates to correct or complete rows where the station identifier is missing.

It is important to consider longitude and latitude as a pair rather than selecting the most frequent longitude and latitude independently. Selecting them separately could produce a coordinate combination that does not actually exist in the original dataset.

The resulting DataFrame contains one representative longitude–latitude pair for each station, together with its frequency of occurrence.

These reference coordinates can then be merged back into the original dataset:

This approach reduces coordinate inconsistencies and provides a stable geographical reference for each meteorological station.

*Special thanks to my colleague Marouane for pointing out this important data-quality issue. His observation helped identify the coordinate inconsistencies associated with each station and guided the development of a more reliable method for assigning representative longitude and latitude values.*



## Comprehensive Analysis and Comparison After Duplicate Removal / Station Coordinate Consistency 

In [ ]:
clean.duplicated().sum() # Check for duplicates in the cleaned dataset

np.int64(0)

In [ ]:
clean["station"].unique() # Check the unique stations in the cleaned dataset if the NaN still exist .

array([60155, 60210, 60135, 60136, 60191, 60156, 60338, 60340, 60060,
       60033, 60200, 60265, 60101, 60285, 60220, 60160, 60120, 60252,
       60150, 60141, 60096, 60280, 60107, 60250, 60115, 60230])

In [ ]:
station_coordinates1 = (
    clean.groupby("station")[["longitude", "latitude"]]
      .nunique()
)

station_coordinates1 # Display the unique longitude and latitude for each station in the cleaned dataset


,longitude,latitude
station,,
60033,1,1
60060,1,1
60096,1,1
60101,1,1
60107,1,1
60115,2,1
60120,1,1
60135,1,1
60136,1,1


### Conclusion

All the structural and geolocation issues identified during the exploratory analysis have been successfully resolved. The dataset was cleaned, validated, and transformed into a consistent CSV format suitable for the subsequent stages of the pipeline.

Several Python scripts were developed to automate the preprocessing workflow, including the conversion of the original Datasetfinal file into CSV format, the validation of the dataset structure, the computation of reference station coordinates, the distance-based filtering of observations, and the removal of duplicate records. These scripts are organized in the src/ directory to ensure a clear, modular, and reproducible preprocessing pipeline.

Finally, the cleaned dataset was validated through a geographic visualization on a map of Morocco. This final verification confirmed that the retained observations are correctly associated with their corresponding reference stations and that the identified anomalies have been successfully corrected .